In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [2]:
# GPU 환경 확인
print("=" * 60)
print("[환경 확인]")
print(f"  CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU 이름: {torch.cuda.get_device_name()}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM 총량: {vram_total:.1f} GB")
    torch.cuda.empty_cache()
print("=" * 60)

[환경 확인]
  CUDA 사용 가능: True
  GPU 이름: NVIDIA GeForce RTX 4060
  VRAM 총량: 8.0 GB


In [3]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

In [4]:
# 모델 & 토크나이저 로드
print(f"모델 로딩 중: {MODEL_ID}")
print("FP16 모드로 로드합니다.")
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # FP16으로 VRAM 절약
    device_map="auto",
)
model.eval()

if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / (1024 ** 3)
    print(f"모델 로드 완료 (현재 VRAM 사용량: {vram_used:.2f} GB)")
else:
    print("모델 로드 완료 (CPU 모드)")

모델 로딩 중: Qwen/Qwen2.5-3B-Instruct
FP16 모드로 로드합니다.


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

모델 로드 완료 (현재 VRAM 사용량: 5.75 GB)


In [5]:
# 1. interactive_chat
def run_interactive_chat():
    print("=" * 60)
    print("[1] interactive_chat")
    print("    'exit', 'quit', '종료' 입력 시 종료")
    print("=" * 60)
 
    # Qwen2.5의 채팅 형식: system, user, assistant 역할 사용
    chat_history = [
        {
            "role": "system",
            "content": (
                "당신은 기계공학 전문가 AI 어시스턴트입니다. "
                "유체역학, 열역학, 재료역학 등 기계공학 전반에 대해 "
                "정확하고 친절하게 답변합니다. 한국어로 답변하세요."
            ),
        },
    ]
 
    while True:
        user_input = input("사용자: ")
 
        if user_input.lower().strip() in ["exit", "quit", "종료"]:
            print("종료합니다.")
            break
 
        # 대화 기록에 추가
        chat_history.append({"role": "user", "content": user_input})
 
        # apply_chat_template으로 토큰나이징
        inputs = tokenizer.apply_chat_template(
            chat_history,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
 
        # 응답 생성
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                max_new_tokens=500,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
            )
 
        # 입력 부분을 제외한 생성된 텍스트만 디코딩
        input_len = inputs["input_ids"].shape[1]
        ai_response = tokenizer.decode(
            output_ids[0][input_len:], skip_special_tokens=True
        ).strip()
 
        print(f"AI: {ai_response}")
        chat_history.append({"role": "assistant", "content": ai_response})
 
        # 대화 기록이 너무 길어지면 오래된 대화 정리 (VRAM 관리)
        # system 메시지(1개) + 최근 10개 메시지(5턴) 유지
        MAX_MESSAGES = 11
        if len(chat_history) > MAX_MESSAGES:
            system_msg = chat_history[0]
            chat_history = [system_msg] + chat_history[-(MAX_MESSAGES - 1):]
 
    print("[대화 기록]")
    for msg in chat_history:
        print(f"  [{msg['role']}] {msg['content'][:80]}...")

In [7]:
run_interactive_chat()

[1] interactive_chat
    'exit', 'quit', '종료' 입력 시 종료


사용자:  안녕


AI: 안녕하세요! 무엇을 도와드릴까요?


사용자:  기계공학의 기초 역학에 대해서 간략하게 알려줘


AI: 네, 기계공학의 기초 역학에 대해 설명드리겠습니다.

기계공학의 기본 개념 중 하나인 역학은 물질의 움직임과 그 요소들 사이의 관계를 연구하는 분야입니다. 주요 측면으로는 다음과 같은 것들이 있습니다:

1. **유동성**: 유체(가스와 액체)의 움직임과 흐름 특성을 이해하는 것을 포함합니다.
2. **회전력 및 축력**: 회전부품이 발생시키는 힘을 분석합니다.
3. **중력**: 물체가 떨어지는 방식이나 지구를 둘러싼 공이 이동하는 방식 등을 이해하는 것입니다.
4. **운동 법칙**: 에탈-뉴턴 운동 법칙과 같이 물리적 현상을 설명하는 기본적인 운동 법칙들을 배웁니다.

이러한 원리를 기반으로 기계 설계와 작동 동작을 예측하고 개선하는데 활용됩니다. 이러한 원리는 다양한 부품들의 동작을 이해하고 조립하여 전체 기계의 성능을 최적화하는데 중요하게 사용됩니다.


사용자:  운동법칙에 대해서 자세히 알려줘


AI: 물론입니다. 운동 법칙은 기계공학에서 매우 중요한 개념 중 하나이며, 특히 에탈-뉴턴 운동 법칙과 가속도 법칙이 가장 잘 알려진 두 가지 입니다. 

### 1. 에탈-뉴턴 운동 법칙 (Newton's Laws of Motion)
에탈-뉴턴 운동 법칙은 물체가 어떤 상태에서 다른 상태로 변화할 때 필요한 힘과 변형된 속도를 설명합니다. 이 법칙은 세 가지로 나뉩니다:

#### 1.1 첫 번째 운동 법칙 (Law of Inertia)
이 법칙은 "모든 물체는 자연스럽게 그대로 있게 유지하려고 한다"라는 내용을 담고 있습니다. 즉, 무언가가 멈추거나 움직일 때까지는 그 상태를 유지하려 합니다. 이를 통해 물체의 운동 방향이나 속도가 변하지 않도록 만드는 것이 가능해집니다.

#### 1.2 두 번째 운동 법칙 (F = ma)
두 번째 운동 법칙은 "모든 물체에는 그 물체의 질량(m)과 가속도(a) 사이에 일정한 비율이 있다"는 내용을 담고 있습니다. 이 법칙은 힘이 곧 질량과 가속도의 곱이라는 의미를 전달합니다. 이를 수학적으로 표현하면 F = ma로 나타낼 수 있습니다.

여기서:
- F: 힘 (N - Newton)
- m: 질량 (kg)
- a: 가속도 (m/s²)

이 법칙을 이용하면 물체가 얼마나 가속되거나 얼마나 느려지게 될지를 계산할 수 있습니다.

#### 1.3 세 번째 운동 법칙 (Conservation of Momentum)
세 번째 운동 법칙은 "양쪽 모두 상호 작용하는 물체들은 그 양측의 모멘텀이 항상 같아져야 한다."는 내용을 담고 있습니다. 즉, 한 쪽이 가속되면 반대편에서도 같은 범위 내에서 마찬가지로 가속되는 것을 의미합니다.

### 2. 가속도 법칙 (Acceleration Law)
가속도 법칙은 에탈-뉴턴


사용자:  quit


종료합니다.
[대화 기록]
  [system] 당신은 기계공학 전문가 AI 어시스턴트입니다. 유체역학, 열역학, 재료역학 등 기계공학 전반에 대해 정확하고 친절하게 답변합니다. 한국어로 답변...
  [user] 안녕...
  [assistant] 안녕하세요! 무엇을 도와드릴까요?...
  [user] 기계공학의 기초 역학에 대해서 간략하게 알려줘...
  [assistant] 네, 기계공학의 기초 역학에 대해 설명드리겠습니다.

기계공학의 기본 개념 중 하나인 역학은 물질의 움직임과 그 요소들 사이의 관계를 연구하는 ...
  [user] 운동법칙에 대해서 자세히 알려줘...
  [assistant] 물론입니다. 운동 법칙은 기계공학에서 매우 중요한 개념 중 하나이며, 특히 에탈-뉴턴 운동 법칙과 가속도 법칙이 가장 잘 알려진 두 가지 입니다...


In [8]:
# 2. single_query
def run_single_query():
    print("\n" + "=" * 60)
    print("[2] single_query")
    print("=" * 60)
 
    messages = [
        {
            "role": "system",
            "content": (
                "당신은 재료역학 전문 튜터입니다. "
                "핵심 개념을 수식과 함께 명확히 설명하고, "
                "실제 설계에서의 적용 사례를 포함하세요."
            ),
        },
        {
            "role": "user",
            "content": (
                "응력-변형률 선도에서 "
                "항복점, 극한강도, "
                "파단점 각각의 물리적 의미와 "
                "기계 설계 시 안전계수와의 관계를 설명해줘."
            ),
        },
    ]
 
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,       # 결정론적 생성 (greedy, 재현성 확보)
            temperature = None,
            eos_token_id=tokenizer.eos_token_id,
        )
 
    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        output_ids[0][input_len:], skip_special_tokens=True
    ).strip()
 
    print(f"[응답]\n{response}")

In [9]:
run_single_query()


[2] single_query


D:\ProgramData\anaconda3\envs\transformer_learn\Lib\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\ProgramData\anaconda3\envs\transformer_learn\Lib\site-packages\transformers\generation\configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[응답]
응력-변형률 선도는 강성과 내구성을 측정하는 중요한 도구로, 이 선도에서의 주요 개념들은 다음과 같습니다.

1. 항복점: 항복점은 강체가 처음으로 압축 또는 힘을 받으면서 변형이 시작되는 응력치입니다. 이 점 이후에 강체는 계속해서 압축 또는 힘을 받으면서 변형이 증가합니다. 항복점은 강체의 내구성을 측정하는데 사용됩니다.

2. 극한강도: 극한강도는 강체가 압축 또는 힘을 받으면서 변형이 일어나지 않는 최대 응력치입니다. 이 점 이후에는 강체가 파괴될 가능성이 있습니다. 극한강도는 강체의 강성을 측정하는데 사용됩니다.

3. 파단점: 파단점은 강체가 완전히 파괴되는 응력치입니다. 이 점 이후에는 강체가 완전히 파괴되어 더 이상의 변형이나 압축/힘을 받을 수 없습니다.

기계 설계에서 안전계수는 이러한 강성과 내구성의 지표들을 이용하여 설계된 기계의 안전성을 평가하는 데 사용됩니다. 안전계수는 강체의 극한강도와 파단점 사이의 비율을 나타내며, 이 값이 클수록 강체가 더 안전하다고 볼 수 있습니다. 예를 들어, 만약 안전계수가 5라면, 강체가 파단점을 초과한 5배의 응력을 받더라도 파괴되지 않을 것입니다.

따라서, 항복점, 극한강도, 파단점은 강체의 성질을 이해하고, 이를 통해 설계된 기계의 안전성을 평가하는데 중요한 역할을 합니다. 이들 개념들은 실제 설계 과정에서 매우 중요하며, 이를 잘 이해하고 적용하면 안전하고 효과적인 기계 설계를 할 수 있습니다.


In [12]:
# 3. pipeline_chat
def run_pipeline_chat():
    print("=" * 60)
    print("[3] pipeline_chat")
    print("=" * 60)
 
    # Pipeline 생성
    chat_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False,    # 생성된 부분만 반환
    )
 
    # 대화 기록 초기화
    conversation_history = [
        {
            "role": "system",
            "content": (
                "당신은 PHM(Prognostics and Health Management) 분야의 "
                "진동 신호 분석 전문가입니다. 회전 기계의 이상치 탐지와 "
                "FFT 분석에 대해 정확하고 실용적으로 설명합니다. "
                "한국어로 답변하세요."
            ),
        },
    ]
    
    # 대화 수행 및 기록 관리
    def chat(user_input: str) -> str:
        conversation_history.append({"role": "user", "content": user_input})
 
        # Pipeline에 대화 기록 전달 -> 내부에서 chat template 적용
        output = chat_pipeline(conversation_history)
        response = output[0]["generated_text"].strip()
 
        conversation_history.append({"role": "assistant", "content": response})
 
        # 대화 기록 길이 관리 (system + 최근 10개 메시지)
        MAX_TURNS = 11
        if len(conversation_history) > MAX_TURNS:
            system_msg = conversation_history[0]
            recent = conversation_history[-(MAX_TURNS - 1):]
            conversation_history.clear()
            conversation_history.append(system_msg)
            conversation_history.extend(recent)
 
        return response
 
    # 예시 대화
    print("--- 질문 1 ---")
    resp1 = chat(
        "회전 기계의 진동 시계열 신호에서 FFT를 이용하여 "
        "이상치를 탐지하는 기본 원리를 설명해줘. "
        "정상 상태와 이상 상태의 주파수 스펙트럼이 어떻게 다른지, "
        "그리고 왜 시간 영역보다 주파수 영역 분석이 결함 탐지에 유리한지도 포함해줘."
    )
    print(f"AI: {resp1}")
 
    print("--- 질문 2 ---")
    resp2 = chat(
        "그러면 FFT 스펙트럼에서 구체적으로 어떤 패턴이 나타날 때 "
        "이상치로 판단해? 예를 들어 특정 주파수에서 피크가 얼마나 커야 "
        "이상으로 볼 수 있는지, 하모닉이나 사이드밴드 같은 패턴은 "
        "각각 어떤 종류의 결함을 의미하는지 설명해줘."
    )
    print(f"AI: {resp2}")
 
    print("[대화 기록 요약]")
    for msg in conversation_history:
        role = msg["role"]
        content = msg["content"][:60]
        print(f"  [{role}] {content}...")

In [13]:
run_pipeline_chat()

Device set to use cuda:0


[3] pipeline_chat
--- 질문 1 ---
AI: 회전 기계의 진동 신호에서 FFT(Fast Fourier Transform)을 사용하여 이상치를 탐지는 다음과 같은 과정으로 이루어집니다.

1. **진동 데이터 수집**: 먼저 회전 기계의 진동 신호를 측정합니다. 이는 일반적으로 특정 위치에서 장치의 부품에 장착된 센서로부터 얻습니다.
2. **FFT 계산**: 다음으로 수집된 진동 신호 데이터를 FFT 알고리즘을 통해 변환합니다. 이 과정에서는 입력 신호를 고주파성과 저주파성을 갖는 부분으로 나누어 주파수 영역에서 표현하게 됩니다.
3. **스펙트럼 해석**: FFT 결과로 얻은 주파수 스펙트럼을 분석합니다. 정상적인 상태에서는 주파수가 일정하거나 가변적이지만, 특정 주파수에 집중되는 경우 이상 상태를 나타낼 수 있습니다. 이러한 특징들은 보통 '불규칙성'이나 '저주파 진동'(저주파로는 0.5Hz~5Hz) 등으로 인식됩니다.

**정상 상태와 이상 상태의 주파수 스펙트럼 차이점**
- **정상 상태**: 일반적으로 고주파 성분과 저주파 성분이 모두 존재하며, 대부분의 주파수에 걸쳐 균형 잡힌 형태의 높은 신호 강도를 보입니다.
- **이상 상태**: 고주파 성분은 약간 감소할 수 있지만, 주로 저주파 성분이 증가하여 고주파 성분이 크게 감소하는 경향이 있습니다. 특히, 특정 주파수에 강력히 집중되는 불규칙한 진동 패턴이 나타날 수 있습니다.

**주파수 영역 분석이 시간 영역 분석보다 유리한 이유**
- **고주파 성분 추적**: 시간 영역에서는 시간에 따른 변화 패턴을 모니터링하는데 효과적이지만, 저주파 성분 또는 고주파 성분의 변화를 명확히 파악하기 어렵습니다. 반면 주파수 영역에서는 주파수가 명확하게 표시되기 때문에, 특정 주파수에서의 변화를 쉽게 식별할 수 있습니다.
- **특정
--- 질문 2 ---
AI: FFT 스펙트럼에서 특정 패턴이 발생하면 이를 이상치나 결함을 알리는 중요한 기준으로 활용할 수 있습니다. 여기서는 주로 고주파 

In [21]:
# 4. fewshot_classification
def run_fewshot_classification():
    print("=" * 60)
    print("[4] fewshot_classification")
    print("=" * 60)
 
    classification_question = (
        "다음 각각의 베어링 진동 분석 결과를 내륜 결함(I), 외륜 결함(O), "
        "볼 결함(B), 정상(N) 중 하나로 분류하라.\n"
        "1) BPFI 2차~5차 하모닉에서 뚜렷한 피크 관측, RMS = 0.045\n"
        "2) 전 주파수 대역에서 에너지가 균일하게 낮음, RMS = 0.008\n"
        "3) BPFO 1차 및 3차 하모닉에서 높은 에너지, RMS = 0.062"
    )
 
    # Zero-shot
    print("[4a] Zero-shot 분류")
    messages_zero = [
        {"role": "user", "content": classification_question},
    ]
 
    inputs = tokenizer.apply_chat_template(
        messages_zero,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )
 
    input_len = inputs["input_ids"].shape[1]
    response_zero = tokenizer.decode(
        output_ids[0][input_len:], skip_special_tokens=True
    ).strip()
    print(f"응답:\n{response_zero}")
 
    # Few-shot
    print("-" * 40)
    print("[4b] Few-shot")
    messages_few = [
        {
            "role": "system",
            "content": (
                "당신은 베어링 진동 분석 전문가다. "
                "FFT 스펙트럼 분석 결과를 바탕으로 베어링 결함 위치를 분류한다. "
                "BPFI는 내륜 통과 주파수, BPFO는 외륜 통과 주파수, "
                "BSF는 볼 스핀 주파수를 의미한다."
            ),
        },
        # Few-shot 예시 1: 외륜 결함
        {
            "role": "user",
            "content": (
                "상황: BPFO 하모닉(1차, 2차, 4차)에서 에너지 집중, "
                "RMS = 0.051, 다른 결함 주파수에서는 특이사항 없음"
            ),
        },
        {
            "role": "assistant",
            "content": (
                "분류: 외륜 결함 (O)\n"
                "이유: BPFO(외륜 통과 주파수)의 하모닉 성분에서만 에너지가 "
                "집중되어 있으므로 외륜에 국부적 결함(점식, 박리 등)이 존재함. "
                "RMS 0.051은 ISO 10816 기준 '주의' 수준에 해당하여, "
                "결함이 초기~중기 단계에 있음을 시사함."
            ),
        },
        # Few-shot 예시 2: 정상
        {
            "role": "user",
            "content": (
                "상황: 전 주파수 대역에서 에너지가 균일하게 낮고 "
                "특정 결함 주파수에서 피크 없음, RMS = 0.006"
            ),
        },
        {
            "role": "assistant",
            "content": (
                "분류: 정상 (N)\n"
                "이유: BPFI, BPFO, BSF 어느 결함 주파수에서도 유의미한 피크가 "
                "관측되지 않으며, RMS 0.006은 정상 운전 범위 내에 있음. "
                "스펙트럼이 전 대역에서 평탄한 노이즈 플로어 수준을 유지하므로 "
                "베어링은 양호한 상태로 판단됨."
            ),
        },
        # 실제 분류 질문
        {"role": "user", "content": classification_question},
    ]
 
    inputs = tokenizer.apply_chat_template(
        messages_few,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )
 
    input_len = inputs["input_ids"].shape[1]
    response_few = tokenizer.decode(
        output_ids[0][input_len:], skip_special_tokens=True
    ).strip()
    print(f"응답:\n{response_few}")

In [22]:
run_fewshot_classification()

[4] fewshot_classification
[4a] Zero-shot 분류
응답:
이 문제는 베어링의 진동 패턴을 통해 결함 유무를 판단하는 방법론을 적용한 것으로 보입니다. 각 베어링 진동 분석 결과를 분석해보겠습니다.

1) BPFI 2차~5차 하모닉에서 뚜렷한 피크 관측, RMS = 0.045

- 이 결과는 다중결함이 있는 경우에 나타날 수 있습니다. 하지만 2차~5차 하모닉에서 뚜렷한 피크가 관찰되므로, 이는 다중결함이 아닌 특정 결함의 증거일 가능성이 높습니다. 특히, 2차 하모닉은 일반적으로 다중결함에서 관찰되는 특징입니다. 따라서 이 경우 I(내륜 결함)를 추정할 수 있습니다.

2) 전 주파수 대역에서 에너지가 균일하게 낮음, RMS = 0.008

- 이 결과는 정상 상태를 나타냅니다. 에너지가 균일하게 낮고, 주파수가 균일하다면, 베어링이 정상 상태임을 의미합니다. 따라서 이 경우 N(정상)을 추정할 수 있습니다.

3) BPFO 1차 및 3차 하모닉에서 높은 에너지, RMS = 0.062

- 이 결과는 외륜 결함(O)을 나타낼 수 있습니다. 외륜 결함에서는 주로 1차 및 3차 하모닉에서 높은 에너지가 관찰됩니다. 따라서 이 경우 O(외륜 결함)을 추정할 수 있습니다.

따라서 각 결과를 바탕으로 다음과 같이 분류할 수 있습니다:

1) I (내륜 결함)
2) N (정상)
3) O (외륜 결함)
----------------------------------------
[4b] Few-shot
응답:
1) 외륜 결함 (O)
2) 정상 (N)
3) 내륜 결함 (I)

이 분석 결과를 설명해 드리면:

1) BPFI(내륜 통과 주파수)의 2차~5차 하모닉에서 뚜렷한 피크가 관찰되므로 내륜에 결함이 존재합니다. RMS 값 0.045은 ISO 10816 기준 '주의' 수준에 해당하며, 이는 결함이 초기 단계일 가능성이 높음을 나타냅니다.

2) 전 주파수 대역에서 에너지가 균일하게 낮으며, RMS 값이 0.008으로 매우 낮습니다. 이는 정상

In [14]:
if torch.cuda.is_available():
    vram_peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"최대 VRAM 사용량: {vram_peak:.2f} GB")

최대 VRAM 사용량: 6.33 GB
